# Google Drive

In [1]:
# # G D R I V E
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


# Installations

In [2]:
# BioPython (Data Gathering)
!pip install biopython
!pip install --upgrade biopython

     |████████████████████████████████| 2.3MB 11.8MB/s 
Requirement already up-to-date: biopython in /usr/local/lib/python3.7/dist-packages (1.78)


In [3]:
# Libraries
from Bio import SeqIO
import numpy as np
import pandas as pd
import os, glob
import seaborn as sns
import matplotlib.pyplot as plt
from multiprocessing import Pool
from tqdm import tqdm
from itertools import groupby
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.feature_extraction.text import CountVectorizer
import math
import json
from functools import partial
from collections import defaultdict

In [4]:
tqdm = partial(tqdm, position=0, leave=True)

# Data Gathering

In [6]:
# Directories
folder_path = "/content/drive/MyDrive/semantic_browser/COVID/Metadata_07-03-2021/"
clades_folders = os.listdir(folder_path)
clades_folders = [x for x in clades_folders if ".gdoc" not in x]
clades_folders = [x for x in clades_folders if ".ipynb_checkpoints" not in x]
clades_folders

['G', 'GR', 'GH', 'GV', 'L', 'S', 'O', 'V', 'GRY']

In [7]:
clade = []
seq = []
id = []
size = []
location = []

for x in clades_folders:
  archive_list = os.listdir(folder_path + x)
  #print(archive_list)# total files/clade folder
  for k in archive_list:
    filename = folder_path + x + "/" + k
    #print(filename) # all file pathways
    #print(k) # all filenames
    for seq_record in SeqIO.parse(filename, "fasta"):
      clade.append(x)
      seq.append(str(seq_record.seq)) 
      id.append(seq_record.id)
      size.append(len(seq_record))
      location.append(k)

In [8]:
# Generate Pandas DataFrame from list of dictionaries
dictionary = {'id': id, 'seq': seq, 'clade': clade, \
              'location': location, 'size': size}
Data_Frame = pd.DataFrame.from_dict(dictionary)

In [9]:
Data_Frame

,id,seq,clade,location,size
0,hCoV-19/Aruba/AW-RIVM-11761/2021|EPI_ISL_10138...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
1,hCoV-19/Aruba/AW-RIVM-11767/2021|EPI_ISL_10138...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
2,hCoV-19/Aruba/AW-RIVM-11740/2021|EPI_ISL_10146...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
3,hCoV-19/Aruba/AW-RIVM-11736/2021|EPI_ISL_10146...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
4,hCoV-19/Aruba/AW-RIVM-11610/2021|EPI_ISL_10146...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
...,...,...,...,...,...
9466,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763
9467,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763
9468,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763
9469,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763


In [10]:
Data_Frame.to_csv('COVID_LATAM_complete.csv')

# Data Import

In [11]:
Data_Frame = pd.read_csv("/content/drive/MyDrive/semantic_browser/COVID/"\
                         "Metadata_07-03-2021/COVID_LATAM_complete.csv", index_col=0)
Data_Frame

,id,seq,clade,location,size
0,hCoV-19/Aruba/AW-RIVM-11761/2021|EPI_ISL_10138...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
1,hCoV-19/Aruba/AW-RIVM-11767/2021|EPI_ISL_10138...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
2,hCoV-19/Aruba/AW-RIVM-11740/2021|EPI_ISL_10146...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
3,hCoV-19/Aruba/AW-RIVM-11736/2021|EPI_ISL_10146...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
4,hCoV-19/Aruba/AW-RIVM-11610/2021|EPI_ISL_10146...,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,G,G_South_America.fasta,29782
...,...,...,...,...,...
9466,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763
9467,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763
9468,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763
9469,hCoV-19/Sint,AGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC...,GRY,GRY_Sint_Marteen.fasta,29763


In [25]:
# Filter by location
Data_Frame['mask'] = Data_Frame['location'].str.contains('V_') 
len(Data_Frame.loc[(Data_Frame['mask'] == True)])
#Data_Frame.drop(['mask'], axis=1)
#print(Data_Frame.iloc[0])

69

# K-mer Decomposition

In [27]:
def k_mer_decomposition(Nucleotide_Sequence, k): # Function to decomposition into k-mers
    Pre_Vocabulary = list(Nucleotide_Sequence[0+i:k+i] for i in range(0, len(Nucleotide_Sequence)- k + 1))
    Extended_Nomenclature_Nucleotides = ['U', 'W', 'S', 'M', 'K', 'R', 'Y', 'B', 'D', 'H', 'V', 'N']
    # To remove the nucleotide extended nomenclature 
    Vocabulary = [words for words in Pre_Vocabulary if all(Nucleotide not in words for Nucleotide in Extended_Nomenclature_Nucleotides)] 
    return Vocabulary

In [28]:
# k-mers path
PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/Metadata_07-03-2021/k-mers/"

In [29]:
k_mer_size = 9
#k_mers = []

batch = 1000
int_value = math.floor(len(Data_Frame)/batch)
for k in tqdm(range(int_value)):
  k_mers = []
  start = k*batch
  end = k*batch+batch
  for x in tqdm(range(start,end)):
    k_mers_list = k_mer_decomposition(Data_Frame['seq'].iloc[x], k_mer_size) # Decompose the genome into overlapped k-mers
    k_mers_string = ' '.join(k_mers_list)
    k_mers.append(k_mers_string)
  with open(PATH_files + str(x) + "_kmer.txt", "w") as fp:   #Serialize
    json.dump(k_mers, fp)

k_mers = []
for a in tqdm(range(end,end+len(Data_Frame)-x)):
  k_mers_list = k_mer_decomposition(Data_Frame['seq'].iloc[x], k_mer_size) # Decompose the genome into overlapped k-mers
  k_mers_string = ' '.join(k_mers_list)
  k_mers.append(k_mers_string)
with open(PATH_files + str(a) + "_kmer.txt", "w") as fp:   #Serialize
  json.dump(k_mers, fp)

100%|██████████| 472/472 [00:21<00:00, 21.58it/s]


# K-mer files into DataFrame

In [ ]:
# Directories
folder_path = "/content/drive/MyDrive/semantic_browser/COVID/k_mers"
file_list = os.listdir(folder_path)
file_list = [x for x in file_list if ".ipynb_checkpoints" not in x]
file_list

['999_kmer.txt',
 '1999_kmer.txt',
 '2999_kmer.txt',
 '3999_kmer.txt',
 '4999_kmer.txt',
 '5999_kmer.txt',
 '6999_kmer.txt',
 '7999_kmer.txt',
 '8999_kmer.txt',
 '9999_kmer.txt',
 '10999_kmer.txt',
 '11999_kmer.txt',
 '12999_kmer.txt',
 '13999_kmer.txt',
 '14999_kmer.txt',
 '15999_kmer.txt',
 '16999_kmer.txt',
 '17999_kmer.txt',
 '18999_kmer.txt',
 '19999_kmer.txt',
 '20999_kmer.txt',
 '21999_kmer.txt',
 '22999_kmer.txt',
 '23999_kmer.txt',
 '24999_kmer.txt',
 '25999_kmer.txt',
 '26999_kmer.txt',
 '27999_kmer.txt',
 '28999_kmer.txt',
 '29999_kmer.txt',
 '30999_kmer.txt',
 '31999_kmer.txt',
 '32999_kmer.txt',
 '33999_kmer.txt',
 '34999_kmer.txt',
 '35999_kmer.txt',
 '36999_kmer.txt',
 '37999_kmer.txt',
 '38999_kmer.txt',
 '39999_kmer.txt',
 '40999_kmer.txt',
 '41999_kmer.txt',
 '42999_kmer.txt',
 '43999_kmer.txt',
 '44999_kmer.txt',
 '45999_kmer.txt',
 '46999_kmer.txt',
 '47999_kmer.txt',
 '48999_kmer.txt',
 '49999_kmer.txt',
 '50999_kmer.txt',
 '51999_kmer.txt',
 '52999_kmer.txt',
 '53

In [ ]:
# Get all filenames
PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/"
filenames = []
for x in tqdm(file_list[0:3]):
  filenames.append(folder_path + "/" + x)

# Create iterable object (generator)
def read_using_generator(filenames):
  read_file = []
  for i in tqdm(filenames):
    with open(i, "r") as fp:   # Unpickling
      read_file_aux = json.load(fp)
    read_file += read_file_aux
  yield (read_file)

read_files = read_using_generator(filenames)

k_mer_list = []
for kmer_file in tqdm(read_files):
  k_mer_list += kmer_file


100%|██████████| 3/3 [00:08<00:00,  2.82s/it]
1it [00:08,  8.47s/it]


In [ ]:
print(k_mer_list[0])

ATTAAAGGT TTAAAGGTT TAAAGGTTT AAAGGTTTA AAGGTTTAT AGGTTTATA GGTTTATAC GTTTATACC TTTATACCT TTATACCTT TATACCTTC ATACCTTCC TACCTTCCC ACCTTCCCA CCTTCCCAG CTTCCCAGG TTCCCAGGT TCCCAGGTA CCCAGGTAA CCAGGTAAC CAGGTAACA AGGTAACAA GGTAACAAA GTAACAAAC TAACAAACC AACAAACCA ACAAACCAA CAAACCAAC AAACCAACC AACCAACCA ACCAACCAA CCAACCAAC CAACCAACT AACCAACTT ACCAACTTT CCAACTTTC CAACTTTCG AACTTTCGA ACTTTCGAT CTTTCGATC TTTCGATCT TTCGATCTC TCGATCTCT CGATCTCTT GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGTAGA CTTGTAGAT TTGTAGATC TGTAGATCT GTAGATCTG TAGATCTGT AGATCTGTT GATCTGTTC ATCTGTTCT TCTGTTCTC CTGTTCTCT TGTTCTCTA GTTCTCTAA TTCTCTAAA TCTCTAAAC CTCTAAACG TCTAAACGA CTAAACGAA TAAACGAAC AAACGAACT AACGAACTT ACGAACTTT CGAACTTTA GAACTTTAA AACTTTAAA ACTTTAAAA CTTTAAAAT TTTAAAATC TTAAAATCT TAAAATCTG AAAATCTGT AAATCTGTG AATCTGTGT ATCTGTGTG TCTGTGTGG CTGTGTGGC TGTGTGGCT GTGTGGCTG TGTGGCTGT GTGGCTGTC TGGCTGTCA GGCTGTCAC GCTGTCACT CTGTCACTC TGTCACTCG GTCACTCGG TCACTCGGC CACTCGGCT ACTCGGCTG CTCGGCTGC TCGGCTGCA CGGCTGCAT 

In [ ]:
# Save k_mer_list in .txt file using json
PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/" # k-mers path
with open(PATH_files + "try1_kmer_list.csv", "r") as fp:   #Serialize
  k_mer_list = json.load(fp)
print(len(k_mer_list))

2000


In [ ]:
PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/"
#k_mer_list = []
filenames = []
for x in tqdm(file_list):
  #k_mer_aux = []
  filenames.append(folder_path + "/" + x)
  #with open(filename, "r") as fp:   # Unpickling
  #  k_mer_aux = json.load(fp)
  #k_mer_list.extend(k_mer_aux)
#with open(PATH_files + "complete_kmer_list.txt", "w") as fp:   #Serialize
#  json.dump(k_mer_list, fp)

100%|██████████| 90/90 [00:00<00:00, 21027.59it/s]


In [ ]:
print(len(filenames))
print(filenames[0])

90
/content/drive/MyDrive/semantic_browser/COVID/k_mers/999_kmer.txt


In [ ]:
kmer_path = "/content/drive/MyDrive/semantic_browser/COVID/complete_kmer_list.txt"
with open(kmer_path, "r") as fp:   # Unpickling
    kmer_list = json.load(fp)

In [ ]:
def read_txt_files(filename):
  """reads the k_mers files with json"""
  filename = folder_path + "/" + filename
  with open(filename, "r") as fp:   # Unpickling
    read_file = json.load(fp)
  k_mers = read_file
  return k_mers

def stack_list(iterable):
    """stack the k_mers in the iterable on a k_mer_list"""
    k_mer_list = []
    for item in iterable:
      k_mer_list += item
    return k_mer_list

def stack_using_generator(files):
  """iterate through a generator and stack the objects"""
  # notice the parentheses instead of square brackets in the next line 
  iterable = (read_txt_files(filename) for filename in tqdm(file_list[0:2]))
  k_mer_complete_list = stack_list(iterable)
  return k_mer_complete_list

k_mers = []
k_mer_list = stack_using_generator(k_mer_list)

# Save k_mer_list in .txt file using json
#PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/" # k-mers path
#with open(PATH_files + "complete_kmer_list.txt", "w") as fp:   #Serialize
#  json.dump(k_mer_list, fp)

NameError: ignored

In [ ]:
PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/"
k_mer_list = []
for x in tqdm(file_list[0:2]):
  k_mer_aux = []
  filename = folder_path + "/" + x
  with open(filename, "r") as fp:   # Unpickling
    k_mer_aux = json.load(fp)
  k_mer_list.extend(k_mer_aux)
#with open(PATH_files + "complete_kmer_list.txt", "w") as fp:   #Serialize
#  json.dump(k_mer_list, fp)
print(k_mer_list[0])

100%|██████████| 2/2 [00:03<00:00,  1.76s/it]

ATTAAAGGT TTAAAGGTT TAAAGGTTT AAAGGTTTA AAGGTTTAT AGGTTTATA GGTTTATAC GTTTATACC TTTATACCT TTATACCTT TATACCTTC ATACCTTCC TACCTTCCC ACCTTCCCA CCTTCCCAG CTTCCCAGG TTCCCAGGT TCCCAGGTA CCCAGGTAA CCAGGTAAC CAGGTAACA AGGTAACAA GGTAACAAA GTAACAAAC TAACAAACC AACAAACCA ACAAACCAA CAAACCAAC AAACCAACC AACCAACCA ACCAACCAA CCAACCAAC CAACCAACT AACCAACTT ACCAACTTT CCAACTTTC CAACTTTCG AACTTTCGA ACTTTCGAT CTTTCGATC TTTCGATCT TTCGATCTC TCGATCTCT CGATCTCTT GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGTAGA CTTGTAGAT TTGTAGATC TGTAGATCT GTAGATCTG TAGATCTGT AGATCTGTT GATCTGTTC ATCTGTTCT TCTGTTCTC CTGTTCTCT TGTTCTCTA GTTCTCTAA TTCTCTAAA TCTCTAAAC CTCTAAACG TCTAAACGA CTAAACGAA TAAACGAAC AAACGAACT AACGAACTT ACGAACTTT CGAACTTTA GAACTTTAA AACTTTAAA ACTTTAAAA CTTTAAAAT TTTAAAATC TTAAAATCT TAAAATCTG AAAATCTGT AAATCTGTG AATCTGTGT ATCTGTGTG TCTGTGTGG CTGTGTGGC TGTGTGGCT GTGTGGCTG TGTGGCTGT GTGGCTGTC TGGCTGTCA GGCTGTCAC GCTGTCACT CTGTCACTC TGTCACTCG GTCACTCGG TCACTCGGC CACTCGGCT ACTCGGCTG CTCGGCTGC TCGGCTGCA CGGCTGCAT 

In [ ]:
PATH_files = "/content/drive/MyDrive/semantic_browser/COVID/"
k_mer_list = []
for x in tqdm(file_list):
  k_mer_aux = []
  filename = folder_path + "/" + x
  with open(filename, "r") as fp:   # Unpickling
    k_mer_aux = json.load(fp)
  k_mer_list.extend(k_mer_aux)
with open(PATH_files + "complete_kmer_list.txt", "w") as fp:   #Serialize
  json.dump(k_mer_list, fp)

 44%|████▍     | 40/90 [04:04<04:50,  5.81s/it]

In [ ]:
PATH = '/content/drive/MyDrive/semantic_browser/COVID/try2_kmer_list.txt'
with open(PATH, "r") as fp:   # Unpickling
    kmer_list_try2 = json.load(fp)

In [ ]:
print(len(kmer_list_try2))
print(kmer_list_try2[0])

3000
ACTTTCGAT CTTTCGATC TTTCGATCT TTCGATCTC TCGATCTCT CGATCTCTT GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGTAGA CTTGTAGAT TTGTAGATC TGTAGATCT GTAGATCTG TAGATCTGT AGATCTGTT GATCTGTTC ATCTGTTCT TCTGTTCTC CTGTTCTCT TGTTCTCTA GTTCTCTAA TTCTCTAAA TCTCTAAAC CTCTAAACG TCTAAACGA CTAAACGAA TAAACGAAC AAACGAACT AACGAACTT ACGAACTTT CGAACTTTA GAACTTTAA AACTTTAAA ACTTTAAAA CTTTAAAAT TTTAAAATC TTAAAATCT TAAAATCTG AAAATCTGT AAATCTGTG AATCTGTGT ATCTGTGTG TCTGTGTGG CTGTGTGGC TGTGTGGCT GTGTGGCTG TGTGGCTGT GTGGCTGTC TGGCTGTCA GGCTGTCAC GCTGTCACT CTGTCACTC TGTCACTCG GTCACTCGG TCACTCGGC CACTCGGCT ACTCGGCTG CTCGGCTGC TCGGCTGCA CGGCTGCAT GGCTGCATG GCTGCATGC CTGCATGCT TGCATGCTT GCATGCTTA CATGCTTAG ATGCTTAGT TGCTTAGTG GCTTAGTGC CTTAGTGCA TTAGTGCAC TAGTGCACT AGTGCACTC GTGCACTCA TGCACTCAC GCACTCACG CACTCACGC ACTCACGCA CTCACGCAG TCACGCAGT CACGCAGTA ACGCAGTAT CGCAGTATA GCAGTATAA CAGTATAAT AGTATAATT GTATAATTA TATAATTAA ATAATTAAT TAATTAATA AATTAATAA ATTAATAAC TTAATAACT TAATAACTA AATAACTAA ATAACTAAT TAACTAATT AACTA

In [ ]:
def stack_images(iterable, n_pixels):
    """stack the images in the iterable on a (n_pixel x n_pixel) canvas"""
    
    canvas = np.zeros((n_pixels,n_pixels))
    
    for item in iterable:
        canvas += item
        
    return canvas

def get_random_image(n_pixels):
    """create a random 2D image with n_pixels on the side"""
    
    return np.random.rand(n_pixels, n_pixels)

def stack_using_generator(n_images, n_pixels):
    """iterate through a generator and stack the objects"""
    
    # notice the parentheses instead of square brackets in the next line 
    iterable = (get_random_image(n_pixels) for _ in range(n_images))
    stack_images(iterable, n_pixels)
    
    # not returning anything because I only care about the memory consumption in creating the iterable

In [ ]:
print(len(k_mer_list))
print(len(k_mer_aux))

3000
1000


In [ ]:
k_mer_list[]

'AGATCTGTT GATCTGTTC ATCTGTTCT TCTGTTCTC CTGTTCTCT TGTTCTCTA GTTCTCTAA TTCTCTAAA TCTCTAAAC CTCTAAACG TCTAAACGA CTAAACGAA TAAACGAAC AAACGAACT AACGAACTT ACGAACTTT CGAACTTTA GAACTTTAA AACTTTAAA ACTTTAAAA CTTTAAAAT TTTAAAATC TTAAAATCT TAAAATCTG AAAATCTGT AAATCTGTG AATCTGTGT ATCTGTGTG TCTGTGTGG CTGTGTGGC TGTGTGGCT GTGTGGCTG TGTGGCTGT GTGGCTGTC TGGCTGTCA GGCTGTCAC GCTGTCACT CTGTCACTC TGTCACTCG GTCACTCGG TCACTCGGC CACTCGGCT ACTCGGCTG CTCGGCTGC TCGGCTGCA CGGCTGCAT GGCTGCATG GCTGCATGC CTGCATGCT TGCATGCTT GCATGCTTA CATGCTTAG ATGCTTAGT TGCTTAGTG GCTTAGTGC CTTAGTGCA TTAGTGCAC TAGTGCACT AGTGCACTC GTGCACTCA TGCACTCAC GCACTCACG CACTCACGC ACTCACGCA CTCACGCAG TCACGCAGT CACGCAGTA ACGCAGTAT CGCAGTATA GCAGTATAA CAGTATAAT AGTATAATT GTATAATTA TATAATTAA ATAATTAAT TAATTAATA AATTAATAA ATTAATAAC TTAATAACT TAATAACTA AATAACTAA ATAACTAAT TAACTAATT AACTAATTA ACTAATTAC CTAATTACT TAATTACTG AATTACTGT ATTACTGTC TTACTGTCG TACTGTCGT ACTGTCGTT CTGTCGTTG TGTCGTTGA GTCGTTGAC TCGTTGACA CGTTGACAG GTTGACAGG TTGACAGGA TGACAGGAC

In [ ]:
frames = [df, df2]
df2 = pd.DataFrame(example2)
result = pd.concat(frames)

In [ ]:
result

,0
0,ATTAAAGGT TTAAAGGTT TAAAGGTTT AAAGGTTTA AAGGTT...
1,ACAAACCAA CAAACCAAC AAACCAACC AACCAACCA ACCAAC...
2,ATTAAAGGT TTAAAGGTT TAAAGGTTT AAAGGTTTA AAGGTT...
3,TTCCCAGGT TCCCAGGTA CCCAGGTAA CCAGGTAAC CAGGTA...
4,GGTTTATAC GTTTATACC TTTATACCT TTATACCTT TATACC...
5,AACTTTAAA ACTTTAAAA CTTTAAAAT TTTAAAATC TTAAAA...
6,CATGCTTAG ATGCTTAGT TGCTTAGTG GCTTAGTGC CTTAGT...
7,GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGT...
8,GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGT...
9,GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGT...


In [ ]:
# Reset index enumeration
result.reset_index(drop=True, inplace=True)

In [ ]:
result.columns = ['try']
result['try'][19]

'TAATAGGTT AATAGGTTT ATAGGTTTA TAGGTTTAT AGGTTTATA GGTTTATAC GTTTATACC TTTATACCT TTATACCTT TATACCTTC ATACCTTCC TACCTTCCC ACCTTCCCA CCTTCCCAG CTTCCCAGG TTCCCAGGT TCCCAGGTA CCCAGGTAA CCAGGTAAC CAGGTAACA AGGTAACAA GGTAACAAA GTAACAAAC TAACAAACC AACAAACCA ACAAACCAA CAAACCAAC AAACCAACC AACCAACCA ACCAACCAA CCAACCAAC CAACCAACT AACCAACTT ACCAACTTT CCAACTTTC CAACTTTCG AACTTTCGA ACTTTCGAT CTTTCGATC TTTCGATCT TTCGATCTC TCGATCTCT CGATCTCTT GATCTCTTG ATCTCTTGT TCTCTTGTA CTCTTGTAG TCTTGTAGA CTTGTAGAT TTGTAGATC TGTAGATCT GTAGATCTG TAGATCTGT AGATCTGTT GATCTGTTC ATCTGTTCT TCTGTTCTC CTGTTCTCT TGTTCTCTA GTTCTCTAA TTCTCTAAA TCTCTAAAC CTCTAAACG TCTAAACGA CTAAACGAA TAAACGAAC AAACGAACT AACGAACTT ACGAACTTT CGAACTTTA GAACTTTAA AACTTTAAA ACTTTAAAA CTTTAAAAT TTTAAAATC TTAAAATCT TAAAATCTG AAAATCTGT AAATCTGTG AATCTGTGT ATCTGTGTG TCTGTGTGG CTGTGTGGC TGTGTGGCT GTGTGGCTG TGTGGCTGT GTGGCTGTC TGGCTGTCA GGCTGTCAC GCTGTCACT CTGTCACTC TGTCACTCG GTCACTCGG TCACTCGGC CACTCGGCT ACTCGGCTG CTCGGCTGC TCGGCTGCA CGGCTGCAT GGCTGCATG

# CountVectorizer

In [ ]:
# Read csv


# Iterate through corpus
# CountVectorizer

# References

* Correct Way To Parse A Fasta File In Python: https://www.biostars.org/p/710/
* Biopython tutorial: https://biopython.org/DIST/docs/tutorial/Tutorial.html#sec12
* SeqIO: https://biopython.org/wiki/SeqIO
* AlignIO: https://biopython.org/wiki/AlignIO
* How to concatenate DataFrames: https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html